# Data Cleaning and Preparation
## Lab 3 — Customer Churn

**Dataset:** A simple Telecom Customer Churn dataset generated using `sklearn.datasets.make_classification`.

### Aim
To clean and prepare customer data for further churn analysis.

### Steps
1. Load the dataset.
2. Explore the data.
3. Handle missing values.
4. Remove duplicates.
5. Standardize text values.
6. Convert data types.
7. Handle simple outliers.
8. Create a new feature.
9. Normalize numerical data.
10. Split into training and testing data.
11. Export the cleaned dataset.


In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)


## 1. Create and load the dataset

`make_classification()` is part of `sklearn.datasets`. It creates a simple classification dataset that we use here as a small telecom churn dataset.


In [2]:
# Create a simple telecom customer dataset using sklearn
X, y = make_classification(
    n_samples=200,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    random_state=42
)

df = pd.DataFrame(X, columns=[
    "Monthly_Charges",
    "Tenure_Months",
    "Support_Calls",
    "Data_Usage"
])

# Make values easier to understand
df["Monthly_Charges"] = (df["Monthly_Charges"] * 20 + 70).round(2)
df["Tenure_Months"] = (df["Tenure_Months"] * 10 + 30).round().astype(int)
df["Support_Calls"] = np.abs(df["Support_Calls"] * 2 + 3).round().astype(int)
df["Data_Usage"] = np.abs(df["Data_Usage"] * 5 + 10).round(2)

df["Contract"] = np.where(
    df["Tenure_Months"] > 36,
    "Long Term",
    "Month-to-Month"
)

df["Churn"] = np.where(y == 1, "Yes", "No")

# Add a few realistic data-quality problems
df.loc[5, "Monthly_Charges"] = np.nan
df.loc[15, "Contract"] = " month-to-month "
df.loc[25, "Contract"] = "LONG TERM"
df.loc[35, "Support_Calls"] = 100

# Add one duplicate row
df = pd.concat([df, df.iloc[[10]]], ignore_index=True)

df.to_csv("Telecom_Customer_Churn.csv", index=False)

print("Dataset created successfully.")
display(df.head())


Dataset created successfully.


,Monthly_Charges,Tenure_Months,Support_Calls,Data_Usage,Contract,Churn
0,74.71,37,3,13.08,Long Term,Yes
1,81.60,40,6,8.18,Long Term,Yes
2,59.58,34,5,10.85,Month-to-Month,Yes
3,43.87,49,0,4.45,Long Term,No
4,63.64,25,2,15.15,Month-to-Month,No


## 2. Explore the dataset

In [3]:
# Shape of the dataset
print("Rows and columns:", df.shape)

# Column names and data types
print("\nData types:")
print(df.dtypes)

# First few records
print("\nFirst 5 records:")
display(df.head())

# Basic statistics
print("\nBasic statistics:")
display(df.describe())


Rows and columns: (201, 6)

Data types:
Monthly_Charges    float64
Tenure_Months        int64
Support_Calls        int64
Data_Usage         float64
Contract               str
Churn                  str
dtype: object

First 5 records:


,Monthly_Charges,Tenure_Months,Support_Calls,Data_Usage,Contract,Churn
0,74.71,37,3,13.08,Long Term,Yes
1,81.60,40,6,8.18,Long Term,Yes
2,59.58,34,5,10.85,Month-to-Month,Yes
3,43.87,49,0,4.45,Long Term,No
4,63.64,25,2,15.15,Month-to-Month,No



Basic statistics:


,Monthly_Charges,Tenure_Months,Support_Calls,Data_Usage
count,200.000000,201.000000,201.000000,201.000000
mean,63.024050,35.109453,3.427861,10.039104
std,24.943834,13.490291,7.337985,4.806134
min,2.450000,0.000000,0.000000,0.370000
25%,47.482500,26.000000,1.000000,7.120000
50%,62.445000,36.000000,2.000000,9.830000
75%,76.527500,46.000000,5.000000,13.160000
max,135.850000,65.000000,100.000000,23.160000


## 3. Check and handle missing values

In [4]:
print("Missing values before cleaning:")
print(df.isnull().sum())

# Fill missing Monthly_Charges with the median
df["Monthly_Charges"] = df["Monthly_Charges"].fillna(
    df["Monthly_Charges"].median()
)

print("\nMissing values after cleaning:")
print(df.isnull().sum())


Missing values before cleaning:
Monthly_Charges    1
Tenure_Months      0
Support_Calls      0
Data_Usage         0
Contract           0
Churn              0
dtype: int64

Missing values after cleaning:
Monthly_Charges    0
Tenure_Months      0
Support_Calls      0
Data_Usage         0
Contract           0
Churn              0
dtype: int64


## 4. Remove duplicate records

In [5]:
print("Duplicate rows before:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows after:", df.duplicated().sum())


Duplicate rows before: 1
Duplicate rows after: 0


## 5. Standardize inconsistent text

In [6]:
# Remove extra spaces and standardize the Contract column
df["Contract"] = df["Contract"].str.strip().str.title()

print(df["Contract"].value_counts())


Contract
Month-To-Month    103
Long Term          97
Name: count, dtype: int64


## 6. Convert columns to correct data types

In [7]:
df["Monthly_Charges"] = pd.to_numeric(df["Monthly_Charges"])
df["Tenure_Months"] = pd.to_numeric(df["Tenure_Months"], downcast="integer")
df["Support_Calls"] = pd.to_numeric(df["Support_Calls"], downcast="integer")
df["Data_Usage"] = pd.to_numeric(df["Data_Usage"])

print(df.dtypes)


Monthly_Charges    float64
Tenure_Months         int8
Support_Calls         int8
Data_Usage         float64
Contract               str
Churn                  str
dtype: object


## 7. Handle a simple outlier

In [8]:
# Support calls should not contain an extremely large value.
# Replace values above 20 with 20.
df["Support_Calls"] = df["Support_Calls"].clip(upper=20)

print("Maximum Support Calls:", df["Support_Calls"].max())


Maximum Support Calls: 20


## 8. Feature engineering

In [9]:
# Create a new feature: estimated yearly charges
df["Yearly_Charges"] = (df["Monthly_Charges"] * 12).round(2)

display(df.head())


,Monthly_Charges,Tenure_Months,Support_Calls,Data_Usage,Contract,Churn,Yearly_Charges
0,74.71,37,3,13.08,Long Term,Yes,896.52
1,81.60,40,6,8.18,Long Term,Yes,979.20
2,59.58,34,5,10.85,Month-To-Month,Yes,714.96
3,43.87,49,0,4.45,Long Term,No,526.44
4,63.64,25,2,15.15,Month-To-Month,No,763.68


## 9. Normalize numerical data

In [10]:
numeric_columns = [
    "Monthly_Charges",
    "Tenure_Months",
    "Support_Calls",
    "Data_Usage",
    "Yearly_Charges"
]

scaler = MinMaxScaler()
df[numeric_columns] = scaler.fit_transform(df[numeric_columns])

display(df.head())


,Monthly_Charges,Tenure_Months,Support_Calls,Data_Usage,Contract,Churn,Yearly_Charges
0,0.541679,0.569231,0.15,0.557701,Long Term,Yes,0.541679
1,0.593328,0.615385,0.30,0.342694,Long Term,Yes,0.593328
2,0.428261,0.523077,0.25,0.459851,Month-To-Month,Yes,0.428261
3,0.310495,0.753846,0.00,0.179026,Long Term,No,0.310495
4,0.458696,0.384615,0.10,0.648530,Month-To-Month,No,0.458696


## 10. Split the data into training and testing sets

In [11]:
# Separate features and target
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Convert categorical Contract column into numeric columns
X = pd.get_dummies(X, columns=["Contract"], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)


Training data: (160, 6)
Testing data : (40, 6)


## 11. Export the cleaned dataset

In [12]:
df.to_csv("Cleaned_Telecom_Customer_Churn.csv", index=False)

print("Cleaned dataset exported successfully.")
display(df.head())


Cleaned dataset exported successfully.


,Monthly_Charges,Tenure_Months,Support_Calls,Data_Usage,Contract,Churn,Yearly_Charges
0,0.541679,0.569231,0.15,0.557701,Long Term,Yes,0.541679
1,0.593328,0.615385,0.30,0.342694,Long Term,Yes,0.593328
2,0.428261,0.523077,0.25,0.459851,Month-To-Month,Yes,0.428261
3,0.310495,0.753846,0.00,0.179026,Long Term,No,0.310495
4,0.458696,0.384615,0.10,0.648530,Month-To-Month,No,0.458696


## Conclusion

The telecom customer data was cleaned and prepared by:
- Handling missing values
- Removing duplicate records
- Standardizing text
- Correcting data types
- Handling a simple outlier
- Creating a new feature
- Normalizing numerical columns
- Splitting the data into training and testing sets
- Exporting the cleaned dataset
